The bulk of this notebook is about building a dataset that maps questions to the _actual_ document chunks that pertain to the question. It starts a little gradio app so that a human can do the labelling.

In [2]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:
import meeplemate
from meeplemate import pdf
from meeplemate.llm_models import (
    load_jina_embedding_model,
    sentence_transformer_to_hf_embeddings,
)
from meeplemate.vectorstores import build_vectorstore_faiss
from pathlib import Path
from langchain.document_loaders import UnstructuredPDFLoader
from langchain.storage import InMemoryByteStore
from langchain.retrievers import MultiVectorRetriever
import uuid
import pandas as pd
from meeplemate.util import (spit_jsonl, slurp_jsonl, spit_json, slurp_json)

/workspaces/mistral-rag-game-rules/code/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/workspaces/mistral-rag-game-rules/code/.venv/lib/python3.10/site-packages/pydantic/_internal/_fields.py:151: UserWarning: Field "model_id" has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(


In [5]:
PROJECT_DIR = Path(meeplemate.__file__).parent.parent
PROJECT_DIR

PosixPath('/workspaces/mistral-rag-game-rules/code')

In [6]:
MAX_DOCUMENT_OPTIONS = 20

In [7]:
DATA_DIR = PROJECT_DIR / "data"
RULES_DIR = DATA_DIR / "rules"
OUTPUT_DIR = DATA_DIR / "output"

In [8]:
jina_embedding_model = load_jina_embedding_model()
hf_embedding_model = sentence_transformer_to_hf_embeddings(jina_embedding_model, normalize_embeddings=True)

/workspaces/mistral-rag-game-rules/code/.venv/lib/python3.10/site-packages/bitsandbytes/cextension.py:34: UserWarning: The installed version of bitsandbytes was compiled without GPU support. 8-bit optimizers, 8-bit multiplication, and GPU quantization are unavailable.
  warn("The installed version of bitsandbytes was compiled without GPU support. "


/workspaces/mistral-rag-game-rules/code/.venv/lib/python3.10/site-packages/bitsandbytes/libbitsandbytes_cpu.so: undefined symbol: cadam32bit_grad_fp32


In [9]:
def get_rule_dirs(base_directory:Path):
    rule_dirs = [d for d in base_directory.iterdir() if (d / "rulebooks.yaml").exists()]
    return rule_dirs

In [10]:
# Get all directories under the rules directory that contian a rulebooks.yaml file
rule_dirs = get_rule_dirs(RULES_DIR)
rule_dirs

[PosixPath('/workspaces/mistral-rag-game-rules/code/data/rules/waterdeep'),
 PosixPath('/workspaces/mistral-rag-game-rules/code/data/rules/pandemic'),
 PosixPath('/workspaces/mistral-rag-game-rules/code/data/rules/wrath_ashardalon'),
 PosixPath('/workspaces/mistral-rag-game-rules/code/data/rules/smallworld'),
 PosixPath('/workspaces/mistral-rag-game-rules/code/data/rules/wyrdwars'),
 PosixPath('/workspaces/mistral-rag-game-rules/code/data/rules/forbiddenisland'),
 PosixPath('/workspaces/mistral-rag-game-rules/code/data/rules/bzbue'),
 PosixPath('/workspaces/mistral-rag-game-rules/code/data/rules/monopoly')]

In [11]:
from meeplemate.llm_models import load_tgi_chat_model
from langchain.text_splitter import RecursiveCharacterTextSplitter

In [12]:
from unstructured.partition.pdf import partition_pdf
rule_doc_path = "../data/rules/pandemic/pandemic_rules2.pdf"
result_ocr = partition_pdf(rule_doc_path, strategy="ocr_only", mode="elements", infer_table_structure=True)
result_fast = partition_pdf(rule_doc_path, strategy="fast", mode="elements", infer_table_structure=True)

In [13]:
# choose player roles, infect your
for i, e in enumerate(result_ocr):
    # print the type in a column of width 20, in a second column print the first 100 characters of the text
    type_name = type(e).__name__

    # Fix any non-ascii characters
    text = e.text.encode('utf-8', 'replace').decode('utf-8')

    text = text[:100]
    print(f"{i:3d} {type_name:20s} {text}")

  0 Title                Pandemic Game Rules
  1 Title                Components
  2 Title                e 5pawns
  3 NarrativeText        e 6research stations e 1 board
  4 Title                e 6 markers
  5 Title                e 96 disease cubes
  6 Title                e 59 player cards
  7 Title                e 48 infection cards
  8 Title                e 5role cards e 4reference cards
  9 Title                Cae aa Imi)
 10 Title                © | suTeReRcs
 11 Title                rx ra or
 12 Title                Ch il DECK
 13 ListItem             1. Place the Board in the center of the table within easy reach of all the players.
 14 ListItem             2. Shuffle the Role cards and deal 1 to each player. Each player takes their corresponding pawn and 
 15 ListItem             3. Place 1 Research Station in Atlanta, and place the others near the side of the board.
 16 ListItem             4. Put the Outbreaks Marker on the "0" space of the Outbreaks Indicator, the Infe

In [14]:
print(result_ocr[38].text)

c) Draw 3 final cards and do the same as above, but add 1 cube to each city


In [15]:
!pip install -U FlagEmbedding

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.4/140.4 KB 5.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for FlagEmbedding: filename=FlagEmbedding-1.2.9-py3-none-any.whl size=165264 sha256=36c6b51b67496eb16cdabd036c186950bbdf7655567bf7057645896c5297ff74
  Stored in directory: /tmp/pip-ephem-wheel-cache-hr83qy_1/wheels/dd/81/52/866c520fb8d78f7ef25ba8463b02796a038655246184b4baad
Successfully built FlagEmbedding
  Attempting uninstall: FlagEmbedding
    Found existing installation: FlagEmbedding 1.2.8
    Uninstalling FlagEmbedding-1.2.8:
      Successfully uninstalled FlagEmbedding-1.2.8


In [16]:
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from FlagEmbedding import FlagReranker

reranker_tokenizer = AutoTokenizer.from_pretrained('BAAI/bge-reranker-large')
reranker_model = FlagReranker('BAAI/bge-reranker-large', use_fp16=True)

pairs = [['what is panda?', 'hi'], ['what is panda?', 'The giant panda (Ailuropoda melanoleuca), sometimes called a panda bear or simply panda, is a bear species endemic to China.']]
with torch.no_grad():
    inputs = reranker_tokenizer(pairs, padding=True, truncation=True, return_tensors='pt', max_length=512)
    scores = reranker_model.compute_score(pairs)
    print(scores)

/workspaces/mistral-rag-game-rules/code/.venv/lib/python3.10/site-packages/torch/cuda/__init__.py:611: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


[-5.608548164367676, 5.762267112731934]


In [17]:
def rerank_documents(query, documents):
    page_contents = [doc.page_content for doc in documents]

    # Get token counts for each document using reranker_tokenizer
    token_counts = [len(reranker_tokenizer.tokenize(page_content)) for page_content in page_contents]

    # bge has a max input length of 512 tokens, so we may need to split the documents into chunks
    text_splitter = text_splitter = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(
        tokenizer=reranker_tokenizer,
        chunk_size=256,
        chunk_overlap=0
    )

    # Split the documents into chunks
    chunks_per_doc = [list(text_splitter.split_text(page_content)) for page_content in page_contents]
    chunk_counts = [len(chunks) for chunks in chunks_per_doc]

    # Flatten the chunks
    flat_chunks = [chunk for chunks in chunks_per_doc for chunk in chunks]

    # Get our pairs for reranking
    pairs = [[query, chunk] for chunk in flat_chunks]

    # Score the pairs
    scores = reranker_model.compute_score(pairs)
    
    # Get the scores for each document
    doc_scores = []
    for i, count in enumerate(chunk_counts):
        doc_scores.append(max(scores[:count]))
        scores = scores[count:]
    
    # Return documents in descending order of score
    documents_sorted = [
        doc 
        for _, doc in sorted(zip(doc_scores, documents), key=lambda pair: pair[0], reverse=True)
    ]

    return documents_sorted

In [18]:
from langchain.schema import Document

testdocs = [Document(page_content='hi'), Document(page_content='The giant panda (Ailuropoda melanoleuca), sometimes called a panda bear or simply panda, is a bear species endemic to China.')]
rerank_documents('what is panda?', testdocs)

[Document(page_content='The giant panda (Ailuropoda melanoleuca), sometimes called a panda bear or simply panda, is a bear species endemic to China.'),
 Document(page_content='hi')]

In [19]:
chat_model = load_tgi_chat_model(inference_server_url="http://tgi:80/", max_new_tokens=2000)

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [20]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

prompt_template = """\
We are converting a PDF scanned using OCR to text. Below is a CURRENT section \
of text from the scanned PDF document. Clean the CURRENT text. This means:

- Fix any spelling errors
- Format the text for clarity
- Add bullet points where needed
- Fix any OCR errors

The cleaned text should be in plain text format and should retain the meaning and content of the CURRENT text.

CURRENT:
{current}

CLEANED TEXT:"""

prompt = ChatPromptTemplate.from_messages(
    [
        ("human", prompt_template)
    ]
)

In [21]:
text_chain = ({"current": RunnablePassthrough()} | prompt | chat_model | StrOutputParser())

In [22]:
import yaml

def copy_curated_questions(directory:Path, output_directory:Path):
    questions_input = directory / "curated_questions.txt"
    questions_output = output_directory / "curated_questions.csv"

    if questions_output.exists():
        print(f"Skipping {questions_output.name}... already converted")
        return

    if not questions_input.exists():
        print(f"Skipping {questions_output.name}... no input file")
        return

    questions = questions_input.read_text().split("\n")
    questions = [q.strip() for q in questions]
    questions = [q for q in questions if q]
    questions_df = pd.DataFrame(questions, columns=["question"])
    questions_df.to_csv(questions_output, index=False)


def convert_rulebooks(directory:Path, output_directory:Path):
    copy_curated_questions(directory, output_directory)

    text_splitter = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(
        tokenizer=chat_model.tokenizer,
        chunk_size=1000,
        chunk_overlap=0
    )
    
    # Load the rulebooks.yaml file
    rulebooks = yaml.safe_load((directory / "rulebooks.yaml").read_text())
    rulebooks_out = []
    for rulebook in rulebooks["rulebooks"]:
        rulebook_name = rulebook["name"]
        rulebook_file = directory / rulebook["path"]
        
        # Get filename of rulebook_file without extension
        rulebook_filename = rulebook_file.stem
        # rulebook_filename = rulebook_name
        outputfile = output_directory / f"{rulebook_filename}.txt"

        rulebooks_out.append({"name": rulebook_name, "path": outputfile.name})

        if outputfile.exists():
            print(f"Skipping {rulebook_name}... already converted")
            continue

        strategy = rulebook.get("strategy", "fast")
        print(f"Converting {rulebook_name}...")

        documents = pdf.parse_pdf(rulebook_file, strategy=strategy)
        document_text = "\n\n".join([i.page_content for i in documents])
        # texts = text_splitter.split_text(document_text)
        # cleaned_texts = text_chain.batch(texts)

        # document_text = "\n\n".join(cleaned_texts)
        outputfile.write_text(document_text)

    rulebook_yaml_out = output_directory / f"rulebooks.yaml"
    rulebook_yaml_out.write_text(yaml.dump({**rulebooks, "rulebooks": rulebooks_out}))


In [23]:
from langchain_core.prompts import ChatPromptTemplate
from meeplemate.question_generation import build_questions_for_documents_chain
from langchain.schema import Document

question_generation_template = """\
You are given an excerpt from a board game rulebook. You're job is to generate \
a list of questions that a player might ask about rules while trying to play \
the game that can be answered by the excerpt. Ensure that the questions are \
clear and concise.

Excerpt:
Once each player has been dealt an envelope, all players should examine their Secret Role cards in secret. Randomly select the first Presidential Candidate and pass that player both the President and Chancellor placards.

For games of 5-6 players, give the following directions to all players:
• Everybody close your eyes.
• Fascist and Hitler, open your eyes and acknowledge each other.
[Take a long pause]
• Everyone close your eyes.
• Everyone can open your eyes. If anyone is confused or something went wrong, please tell the group now.

Questions:
Q1: How is the first Presidential Candidate selected?
Q2: For games of 5-6 players, do Facists and Hitler open their eyes at the same time?
Q3: Should I share what my secret role is?
Q4: If I'm Hitler, do I keep my eyes open or closed along with the facists?
Q5: If I'm not Hitler or a Facist, do I open my eyes at any point?

Excerpt:
ON ELIGIBILITY:
• Term limits apply to the President and Chancellor who were last elected, not to the last pair nominated.  
• Term limits only affect nominations to the Chancellorship; anyone can be President, even someone who was just Chancellor.
• If there are only five players left in the game, only the last elected Chancellor is ineligible to be Chancellor Candidate; the last President may be nominated.
• There are some other rules that affect eligibility in specific ways: the Veto Power and the Election Tracker. You don't need to worry about those yet, and we'll talk about each one in its relevant section.t to the last pair nominated.

Questions:
Q1: Who is affected by term limits?
Q2: Can the last President be nominated as Chancellor?
Q3: Can the last elected Chancellor be nominated as Chancellor again?
Q4: What rules affect eligibility for the Chancellorship and Presidency?
Q5: If more than 5 players are left, can the previous President be nominated as Chancellor?
Q6: Can I be President if I was just Chancellor?
Q7: Can I be Chancellor if I was just President?

Except:
{doc}
"""

question_generation_prompt = ChatPromptTemplate.from_template(
    question_generation_template
)

question_chain = build_questions_for_documents_chain(chat_model, prompt=question_generation_prompt)

rule_except = """\
Sequence of Play
The Lords of Waterdeep game is played in rounds. During each round, players take turns, in which they assign their Agents to various tasks.
The game ends after 8 rounds have passed. The player with the most VP at the end of the eighth round is the winner.
Start of Round
The round spaces on the game board start with 3 VP tokens on each space, as indicated in the setup instructions. The VP tokens also serve to mark which round it is.
At the start of each round, remove the 3 VP tokens from that round's space and place 1 VP token on each face-up Building in Builder's Hall.
When Purchased/Start of Round: Some Building tiles have special instructions to be followed when the Building is purchased and at the start of each round. If any of those Buildings are in play, follow the start-of-round instructions for each. (See Appendix 1: Buildings on page 12 for descriptions of these effects.)
Once all start-of-round effects are complete, players take turns in order.
Start of Round 5: Each player takes the extra Agent piece of his or her color from the space near the rounds track and adds it to his or her pool. The extra Agent is available for the rest of the game.
Actions in a Turn
Each player takes turns, one at a time, starting with the player who has the First Player marker and proceeding to the player's left.
During your turn, if you have Agents available to assign, you take one or both of the following actions:
1. Assign Agent
2. Complete Quest
Assign Agent
If you have any Agents in your pool, you assign 1 of them. To assign an Agent, place it on any unoccupied action space of a Building, whether a basic Building or one that has been put into play. You cannot place an Agent on an action space that contains another Agent (yours or another player's) or on Buildings that are not yet in play.
When you assign an Agent, follow the instructions for that action space. You take that action just once.
You cannot choose to pass your turn. If you have Agents available, you must assign 1 of them. (In the unlikely event that you cannot take an action on your turn, you must pass.)
If you have no more Agents available to assign, you can no longer take turns that round. For the rest of the round, play skips to the next player in order who still has available Agents.
Several basic Buildings have special rules, as described below. For detailed descriptions of all Buildings' effects, see Appendix 1: Buildings on page 12.
"""

question_chain.invoke([Document(page_content=rule_except)])

[['How does the sequence of play work in each round?',
  'What happens at the start of each round?',
  'How do the VP tokens on the game board work?',
  'What is the significance of the extra Agent piece introduced in Round 5?',
  'How do you assign an Agent during your turn?',
  'Can you choose to pass your turn?',
  "What happens if you don't have any Agents available to assign?",
  'What are the two types of actions you can take during your turn?',
  'How do you complete a quest during your turn?',
  'What are the special rules for the basic Buildings?']]

In [24]:
def add_questions(rule_dir:Path):
    questions_csv = rule_dir / "questions.csv"
    if questions_csv.exists():
        print(f"Skipping {rule_dir.name}... questions already present")
        return
    
    all_questions = []
    
    rulebooks = yaml.safe_load((rule_dir / "rulebooks.yaml").read_text())
    game_name = rulebooks["name"]
    for rulebook in rulebooks["rulebooks"]:
        # Skip if questions already present

        rulebook_file = rule_dir / rulebook["path"]
        
        # Read rulebook text
        rulebook_text = rulebook_file.read_text()

        # Make rulebook into a document
        rulebook_doc = Document(page_content=rulebook_text)

        # Split rulebook into chunks
        text_splitter = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(
            tokenizer=chat_model.tokenizer,
            chunk_size=512,
            chunk_overlap=0
        )
        documents = text_splitter.split_documents([rulebook_doc])

        # Generate questions
        questions = question_chain.invoke(list(documents))

        # Flatten questions
        questions_flat = [q for qs in questions for q in qs]

        # Add rulebook name
        questions_flat = [{"game_name": game_name, "name": rulebook["name"], "question": q} for q in questions_flat]
        
        all_questions.extend(questions_flat)

    pd.DataFrame.from_records(all_questions).to_csv(questions_csv, index=False)

In [25]:
for rule_dir in rule_dirs:
    rule_output_dir = OUTPUT_DIR / "rules" / rule_dir.stem
    rule_output_dir.mkdir(exist_ok=True, parents=True)
    convert_rulebooks(rule_dir, rule_output_dir)
    add_questions(rule_output_dir)

Skipping curated_questions.csv... already converted
Skipping Lords of Waterdeep... already converted
Skipping waterdeep... questions already present
Skipping curated_questions.csv... already converted
Skipping Pandemic... already converted
Skipping pandemic... questions already present
Skipping curated_questions.csv... already converted
Skipping Wrath of Ashardalon Rulebook... already converted
Skipping Wrath of Ashardalon Adventure Book... already converted
Skipping wrath_ashardalon... questions already present
Skipping curated_questions.csv... already converted
Skipping Small World... already converted
Skipping smallworld... questions already present
Skipping curated_questions.csv... already converted
Skipping WyrdWars Rulebook... already converted
Skipping WyrdWars Magic Book... already converted
Skipping WyrdWars Special Skills Book... already converted
Skipping WyrdWars Mercenaries Warband Book... already converted
Skipping wyrdwars... questions already present
Skipping curated_qu

In [26]:
def split_documents(documents, chunk_size=500, chunk_overlap=50):
    import hashlib
    text_splitter = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(
        tokenizer=chat_model.tokenizer,
        add_start_index=True,
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )
    split_documents = text_splitter.split_documents(documents)
    for doc in split_documents:
        # Calc sha1 of page_content
        page_content = doc.page_content.encode("utf-8")
        doc.metadata["chunk_id"] = hashlib.sha1(page_content).hexdigest()
    return split_documents


def documents_to_dataframe(documents):
    records = [
        {
            **doc.metadata,
            "page_content": doc.page_content,
        }
        for doc in documents
    ]
    return pd.DataFrame.from_records(records)


def dataframe_to_documents(df):
    from langchain_community.document_loaders import DataFrameLoader
    loader = DataFrameLoader(df, page_content_column="page_content")
    return loader.load()


def create_documents_files(directory:Path):
    documents_file = directory / "documents.csv"
    split_documents_file = directory / "split_documents.csv"
    if split_documents_file.exists():
        print(f"Skipping {directory.name}... documents already present")
        return

    rulebooks = yaml.safe_load((directory / "rulebooks.yaml").read_text())
    game_name = rulebooks["name"]
    documents = []
    for rulebook in rulebooks["rulebooks"]:
        rulebook_file = directory / rulebook["path"]
        rulebook_text = rulebook_file.read_text()
        document = Document(
            page_content=rulebook_text,
            metadata={
                "game_name": game_name,
                "name": rulebook["name"],
                "doc_id": str(uuid.uuid4())
            }
        )
        documents.append(document)
    
    documents_to_dataframe(documents).to_csv(documents_file, index=False)

    child_documents = split_documents(documents)
    documents_to_dataframe(child_documents).to_csv(split_documents_file, index=False)


def load_documents_from_csv(documents_csv:Path):
    df = pd.read_csv(documents_csv)
    return dataframe_to_documents(df)

In [27]:
output_rule_dirs = get_rule_dirs(OUTPUT_DIR / "rules")
for rule_dir in output_rule_dirs:
    print(f"Processing {rule_dir.name}...")
    create_documents_files(rule_dir)

Processing waterdeep...
Skipping waterdeep... documents already present
Processing pandemic...
Skipping pandemic... documents already present
Processing wrath_ashardalon...
Skipping wrath_ashardalon... documents already present
Processing smallworld...
Skipping smallworld... documents already present
Processing wyrdwars...
Skipping wyrdwars... documents already present
Processing forbiddenisland...
Skipping forbiddenisland... documents already present
Processing bzbue...
Skipping bzbue... documents already present
Processing monopoly...
Skipping monopoly... documents already present


In [28]:
def load_questions_dataframe(rule_directory:Path):
    questions_csv = rule_directory / "curated_questions.csv"
    return pd.read_csv(questions_csv)


def load_split_documents(rule_directory:Path):
    split_documents_csv = rule_directory / "split_documents.csv"
    return load_documents_from_csv(split_documents_csv)

In [29]:
def build_retriever(documents, id_key="chunk_id"):
    vectorstore = build_vectorstore_faiss(hf_embedding_model)
    store = InMemoryByteStore()
    search_kwargs={"k": MAX_DOCUMENT_OPTIONS*2, "fetch_k": (MAX_DOCUMENT_OPTIONS*2)+5}
    retriever = MultiVectorRetriever(
        vectorstore=vectorstore,
        byte_store=store,
        id_key=id_key,
        search_kwargs=search_kwargs,
    )

    # Add documents to docstore
    doc_ids = [document.metadata[id_key] for document in documents]
    retriever.docstore.mset(list(zip(doc_ids, documents)))

    # Map question embeddings to their documents
    question_chain = build_questions_for_documents_chain(chat_model)
    hypothetical_questions = question_chain.invoke(documents)
    question_docs = []
    for i, question_list in enumerate(hypothetical_questions):
        question_docs.extend(
            [Document(page_content=question, metadata={id_key: doc_ids[i]}) for question in question_list]
        )
    retriever.vectorstore.add_documents(question_docs)

    # Map document chunk embeddings to their parent documents
    child_splitter = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(
        chat_model.tokenizer,
        chunk_size=125,
        chunk_overlap=12
    )
    for doc_id, document in zip(doc_ids, documents):
        child_docs = child_splitter.split_documents([document])
        for child_doc in child_docs:
            child_doc.metadata[id_key] = doc_id
        retriever.vectorstore.add_documents(child_docs)
    
    return retriever

In [30]:
def snake_case(s):
    return s.lower().replace(" ", "_")

def hash_string(s):
    import hashlib
    return hashlib.md5(s.encode()).hexdigest()

def build_question_retrieved_documents(questions_df, documents, doc_id_key="chunk_id"):
    if isinstance(questions_df, list):
        questions = questions_df
    else:
        questions = questions_df["question"].tolist()

    # Find documents related to questions
    retriever = build_retriever(documents)
    retrieved_documents_per_question = list(retriever.batch(questions))

    # Rerank documents for each question
    retrieved_documents_per_question = [
        rerank_documents(question, retrieved_documents)
        for question, retrieved_documents in zip(questions, retrieved_documents_per_question)
    ]

    # Get top 5 retrieved documents
    retrieved_documents_per_question = [retrieved[:MAX_DOCUMENT_OPTIONS] for retrieved in retrieved_documents_per_question]

    # Build output
    samples = []
    for i, (question, retrieved_documents) in enumerate(zip(questions, retrieved_documents_per_question)):
        game_name = retrieved_documents[0].metadata["game_name"]
        document_dicts = [doc.dict() for doc in retrieved_documents]
        doc_ids = [doc["metadata"][doc_id_key] for doc in document_dicts]
        sample_id = hash_string("#".join([question, *doc_ids]))
        sample_id = f"{snake_case(game_name)}#{sample_id}"
        samples.append(
            {
                "sample_id": sample_id,
                "game_name": game_name,
                "question": question,
                "retrieved_documents": document_dicts,
            }
        )
    
    return samples

In [31]:
def cache_jsonl(path):
    if not isinstance(path, Path):
        path = Path(path)
    
    def decorator(func):
        def wrapper(*args, **kwargs):
            if path.exists():
                return slurp_jsonl(path)
            else:
                data = func(*args, **kwargs)
                spit_jsonl(data, path)
                return data
        return wrapper
    return decorator

In [32]:
from meeplemate.util import spit_jsonl, slurp_jsonl

for rule_dir in output_rule_dirs:
    questions_df = load_questions_dataframe(rule_dir)
    documents = load_split_documents(rule_dir)
    question_retrieved_documents = cache_jsonl(rule_dir / "curated_questions_documents.jsonl")(
        build_question_retrieved_documents
    )(questions_df, documents)

In [33]:
def build_all_question_context(output_rule_base):
    rule_dirs = get_rule_dirs(output_rule_base)
    # Sort rule_dirs
    rule_dirs = sorted(rule_dirs, key=lambda x: str(x))
    all_question_context = []
    for rule_dir in rule_dirs:
        question_retrieved_documents = slurp_jsonl(rule_dir / "curated_questions_documents.jsonl")
        all_question_context.extend(question_retrieved_documents)
    return all_question_context

In [34]:
output_rule_base = OUTPUT_DIR / "rules"
all_question_context = cache_jsonl(output_rule_base / "curated_questions_documents.jsonl")(build_all_question_context)(output_rule_base)

In [35]:
def question_documents_to_choices(question_documents, doc_id_key="chunk_id"):
    choices = [
        (doc["page_content"], doc["metadata"][doc_id_key])
        for doc in question_documents["retrieved_documents"]
    ]
    return choices

In [36]:
question_documents_to_choices(all_question_context[0])

[("**Scoring and Tracking**\n1. Teams are awarded points during their turn by scoring touchdowns and claiming Challenge cards. Each team's score is recorded by the position of their score marker on the score track.\n2. Once a team's score reaches 10 points, their team coin is placed on the +10 space and their score marker is returned to 0. The same happens when their score reaches 20 or 30. So, for example, a team with a score of 17 would have their score marker on the 7 space and their coin on the +10 space.\n\n**Setting Up the Game**\n1. To setup a game of Blitz Bowl, follow the steps outlined below:\n    * First, flip one of the team coins to determine which coach wins the toss. The winning coach chooses the pitch they want to use and places the game board in the centre of the table with that side face up. If this is your first game, we recommend using the pitch with a single trapdoor.\n    * A dugout is placed at each end of the board, as shown in the diagram.\n    * Next, the coac

In [37]:
import gradio as gr



selection_path = OUTPUT_DIR / "rules" / "oracle_documents.json"
if not selection_path.exists():
    spit_json({}, selection_path)

oracle_document_selection = slurp_json(selection_path)

skip_ahead_to_next_unmarked = True

try:
    demo.close()
except:
    pass

gr.close_all()

print("Closed demo")

current_index = -1
question_data = all_question_context

print(oracle_document_selection)

if True:
    for i, question in enumerate(question_data):
        sample_id = question["sample_id"]
        if sample_id in oracle_document_selection:
            current_index = i - 1
            break

print(f"Fastfowarded to {current_index}")

def update_selected(*selected):
    global current_index
    global question_data

    print("here")
    print(selected)

    # Update selected documents
    if current_index > -1:
        sample_id = question_data[current_index]["sample_id"]
        docs_selected = [
            doc["metadata"]["chunk_id"]
            for doc, selected in zip(question_data[current_index]["retrieved_documents"], selected)
            if selected
        ]
        oracle_document_selection[sample_id] = docs_selected
    
    if current_index >= len(question_data):
        current_index = -1
        # return (
        #     gr.Markdown("### Done!"),
        #     gr.Radio(choices=[]),
        #     gr.Button("Restart")
        # )
        return "Done!", "Restart", *checkboxes, *doc_texts

    current_index += 1

    while current_index < len(question_data):
        sample_id = question_data[current_index]["sample_id"]
        if sample_id not in oracle_document_selection:
            break
        current_index += 1

    sample_id = question_data[current_index]["sample_id"]
    selection = oracle_document_selection.get(sample_id, [])
    
    checkboxes = []
    doc_texts = []

    for i in range(MAX_DOCUMENT_OPTIONS):
        doc = None
        if i < len(question_data[current_index]["retrieved_documents"]):
            doc = question_data[current_index]["retrieved_documents"][i]
        
        if doc is None:
            checkboxes.append(gr.Checkbox(value=False, visible=False))
            doc_texts.append(gr.Textbox(f"", visible=False))
        else:
            doc_id = doc["metadata"]["chunk_id"]
            selected = doc_id in selection
            checkboxes.append(gr.Checkbox(value=selected, visible=True))
            doc_texts.append(gr.Textbox(doc["page_content"], visible=True))
    
    print("selection", selection)

    question = question_data[current_index]["question"]
    game_name = question_data[current_index]["game_name"]

    content = f"""\
# {game_name}

## Question

{question}
"""

    return content, "Submit", *checkboxes, *doc_texts

with gr.Blocks() as demo:
    title = gr.Markdown("Ready to start")
    checkboxes = []
    doc_texts = []
    for i in range(MAX_DOCUMENT_OPTIONS):
        checkboxes.append(gr.Checkbox(label=f"document {i}", value=True, visible=False))
        doc_texts.append(gr.Textbox(f"some text {i}", visible=False))
    # doc_texts = gr.CheckboxGroup(label="Documents", choices=[])
    submit_button = gr.Button("Start")
    submit_button.click(
        update_selected, 
        inputs=[*checkboxes], 
        outputs=[title, submit_button, *checkboxes, *doc_texts]
    )

demo.launch(inline=False, server_name="0.0.0.0")

Closed demo
{'blitz_bowl_ultimate_edition#30b08a71c5901b1931d2b0490d26b5fd': ['5caad3e13e34bf3b20e99281687c773302d6479c', '599324af4e48998ea1d83ac25067e3191f3b5a70'], 'blitz_bowl_ultimate_edition#f99b5d2ad9662960b60f58a7e92a5a64': ['599324af4e48998ea1d83ac25067e3191f3b5a70'], 'blitz_bowl_ultimate_edition#f77516fe82316342d8c96c3416b87f36': ['ceca35e7c6cee4f1511af2a475cc4ac7adafe7b4'], 'blitz_bowl_ultimate_edition#3b9a20295d60a9ea573e0084423eac30': ['ceca35e7c6cee4f1511af2a475cc4ac7adafe7b4', 'a33e99a2545b4c6728b33490bb0d72dc4d1edd90'], 'blitz_bowl_ultimate_edition#43a85a3a7a7cdb34e8554b72df4b9316': ['a3866c866cef25d4aa745f0f17c6f4a23798c02b'], 'blitz_bowl_ultimate_edition#262e519719d12331495fd8ea4de58f66': ['a33e99a2545b4c6728b33490bb0d72dc4d1edd90'], 'blitz_bowl_ultimate_edition#31f576d4cbbaeb732d5a0c332d89143f': ['a3866c866cef25d4aa745f0f17c6f4a23798c02b'], 'blitz_bowl_ultimate_edition#488c50b4316d2873503f73e454cea8ea': ['3d07af51a4d3e313178a7950f613f928be358240'], 'blitz_bowl_ultimat

here
(True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True)
selection []
here
(False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False)
selection []
here
(True, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False)
selection []
here
(True, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False)
selection []
here
(False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False)
selection []
here
(True, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False)
selection []


In [37]:
gr.__version__

'4.24.0'

In [38]:
len(question_data[0]['retrieved_documents'])

7

here
(True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True)
selection []


In [40]:
spit_json(oracle_document_selection, selection_path)